# Quantum read-only memory

A QROM preserves its address and XORs the selected classical values into its quantum target words. The same operation reverses the XOR. Several datasets can share one address, even when their target shapes and word widths differ.

Adapted from the Qualtran v0.7.0 QROM tutorial. Copyright 2023 Google LLC. Licensed under Apache-2.0.

In [ ]:
from collections import Counter

import numpy as np
import torch
from IPython.display import SVG, Markdown, display
from sympy import symbols

from examples.algorithms.binary_field_arithmetic import FieldOperation
from examples.algorithms.field_addition_catalog import field_signature_svg
from flagquantum.compiler._component_trace.call_graph import symbolic_call_graph
from flagquantum.compiler.registers.lookup.qrom_resource_model import source_qrom_calls
from flagquantum.compiler.registers.lookup.qrom_table import QromTable
from flagquantum.drawer.call_graph import (
    export_call_graph_dot,
    format_call_graph_markdown,
    render_call_graph_svg,
)
from flagquantum.dynamic import DynamicCircuit
from flagquantum.runtime.dynamic.execution import run_dynamic

## Selection and target shapes

A concrete dataset has its common selection shape followed by its own target shape. The address registers are bounded by the number of entries on each axis. The target qubits are grouped in dataset order, row-major within each target array, with the most significant bit first within a word. Explicit target widths can reserve more bits than the values require. Shape-only declarations can carry symbolic widths and a symbolic number of controls.

The unary traversal computes a predicate for each visited segment. A subtree whose relevant stored values are uniform can reuse a single load. The T-family cost scales with visited branches. Explicit resource declarations retain logical ANDs and their adjoints, while the executable stream contains the instructions that realize them.

[Unary iteration and QROM](https://arxiv.org/abs/1805.03662)

In [ ]:
small = QromTable.from_data(np.arange(5), target_bitsizes=(3,))
multi = QromTable.from_data(np.arange(5), np.arange(5) + 1, target_bitsizes=(3, 4))
grid_data = np.arange(9).reshape(3, 3)
grid = QromTable.from_data(grid_data, grid_data + 1, target_bitsizes=(8, 8))
n, m, b1, b2, controls = symbols("N M b1 b2 c")
unknown = QromTable.from_shape((n, m), (b1, b2), num_controls=controls)
display(SVG(field_signature_svg((
    ("QROM: small", small.signature, {}),
    ("QROM: multiple datasets", multi.signature, {}),
    ("QROM: multiple axes", grid.signature, {}),
    ("QROM: symbolic", unknown.signature, {}),
))))

In [ ]:
shaped = QromTable.from_data(
    [0, 1], [[2, 3], [1, 0]], target_bitsizes=(1, 2), target_shapes=((), (2,))
)
assert shaped.shape == (2,) and shaped.target_layout == ((1, 1), (2, 2))
unknown.resource_formula()

## Direct calls of the five-row table

The graph reports the logical source calls before lowering. Each split uses a negative-control AND, a CNOT to move to its other branch, and a positive adjoint. The outer uncontrolled traversal uses two X calls.

In [ ]:
calls = source_qrom_calls(
    0, small.selection_bitsizes, small.shape, small.target_layout, small.datasets
)
root = ("QROM", (), False)
graph = symbolic_call_graph(root, {root: calls}, max_depth=1)

def label(node):
    name, parameters, adjoint = node
    arguments = f"{parameters}" if parameters else ""
    return name + arguments + ("†" if adjoint else "")

display(SVG(render_call_graph_svg(export_call_graph_dot(root, graph.direct_calls, title=label))))
display(Markdown(format_call_graph_markdown(root, graph.direct_calls, graph.leaf_counts, title=label)))

## Execute through append

Start the five addresses in a coherent state and set the target to one. A loaded word is XORed with one. The caller provides two clean work wires and their feedback slots. The expected vector keeps the original complex amplitude at each loaded address and has zero work wires.

In [ ]:
port_count = small.signature.n_qubits()
work = tuple(range(port_count, port_count + small.workspace_size))
wire_count = port_count + len(work)
body = small.bind_packed(tuple(range(port_count)), work, tuple(range(len(work))))
initial = torch.zeros(2**wire_count, dtype=torch.complex128)
expected = torch.zeros_like(initial)
for row, amplitude in enumerate((1, 1j, -1, -1j, 1)):
    initial[((row << 3) | 1) << len(work)] = amplitude / np.sqrt(5)
    expected[((row << 3) | (1 ^ row)) << len(work)] = amplitude / np.sqrt(5)
circuit = DynamicCircuit(wire_count, inputs=initial, dtype=torch.complex128)
circuit.append(FieldOperation(wire_count, body))
result = run_dynamic(circuit, shots=16, seed=73, strategy="batched")
torch.testing.assert_close(
    result.final_states, expected.expand_as(result.final_states), atol=2e-12, rtol=0
)
{"emitted": dict(Counter(g.name for g in body)), "source_model": small.resource_model().gates}